# 03 — Baseline con Features Combinadas: TF-IDF + LIWC + Polarity + Temporal → LogReg
**Autores:** Giuliano Crenna, Bruno Emmanuel Pace (UGR)  
**Etapa:** 4 — Modelado  

Sube el techo del baseline clásico integrando las features linguísticas (LIWC), polaridad (VADER) y temporales que el EDA ya mostró discriminativas:
- **LIWC** negative_emotion: +62% en depressive (EDA 03)
- **LIWC** positive_emotion: −32% en depressive
- **Polarity** (VADER compound): −83% en depressive (de −0.029 a −0.052)

Se concatenan con TF-IDF vía `ColumnTransformer` (StandardScaler sobre las densas) y se entrena LogReg con los mismos hiperparámetros del baseline. Train downsampled 50/50 (424.436 filas) para comparar en igualdad con `logreg_downsampled`.

In [ ]:
# Parámetros
SEED = 42
DOWNSAMPLE_SEED = 42
OUT_DIR = "reports"

In [ ]:
import os, sys, time
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from src.utils.seeds import set_seed
from src.models.train_baseline import load_config, load_split_data
from src.features.liwc_counts import count_markers
from src.features.polarity import score as polarity_score
from src.features import temporal_features as _tf_mod
from src.evaluation.metrics import compute

# Monkey-patch: temporal_features.extract falla en pandas moderno con .join(on=).
# Reemplazo por una versión equivalente que usa transform() (más robusta).
def _safe_extract(df, ts_col="timestamp"):
    ts = pd.to_datetime(df[ts_col], errors="coerce", utc=True)
    out = pd.DataFrame(index=df.index)
    out["hour_of_day"] = ts.dt.hour.fillna(-1).astype(int)
    out["day_of_week"] = ts.dt.dayofweek.fillna(-1).astype(int)
    out["is_night"] = ((out["hour_of_day"] >= 22) | (out["hour_of_day"] < 6)).astype(int)
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["month"] = ts.dt.month.fillna(-1).astype(int)
    counts = df.groupby("user_id")["user_id"].transform("size")
    out["msg_per_user"] = counts.fillna(0).astype(int).values
    return out
_tf_mod.extract = _safe_extract
from src.features.temporal_features import extract as temporal_extract  # noqa: E402

set_seed(SEED)
OUT_DIR = Path(OUT_DIR)
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
MODEL_DIR = REPO_ROOT / "models"
DATA_PROC = REPO_ROOT / "data" / "processed"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

META_COLS = {
    "doc_id", "user_id", "source", "timestamp", "text_raw", "text_clean",
    "label", "label_source", "lang", "corpus_source",
    "is_translated", "mt_system", "text_orig", "lang_orig",
}
LEX_PATH = REPO_ROOT / "src" / "features" / "lexicons" / "leis_lexicon.csv"
POL_PATH = REPO_ROOT / "src" / "features" / "lexicons" / "polarity_es.csv"

print(f"Repo: {REPO_ROOT}")
print(f"Out: {OUT_DIR.resolve()}")

## 1. Carga de splits user-level

In [ ]:
splits_dir = REPO_ROOT / "data" / "processed" / "splits"
assert (splits_dir / "train.parquet").exists()
df_train, df_val, df_test = load_split_data(data_dir=splits_dir, max_samples=None, seed=SEED)

print(f"Train: {len(df_train):,} | {df_train['label'].value_counts().to_dict()}")
print(f"Val:   {len(df_val):,} | {df_val['label'].value_counts().to_dict()}")
print(f"Test:  {len(df_test):,} | {df_test['label'].value_counts().to_dict()}")
assert len(df_train) == 700717 and len(df_val) == 95630 and len(df_test) == 250847
print("OK — conteos match con split_manifest.json")

## 2. Pre-cómputo de features hand-crafted (cache en parquet)

In [ ]:
def build_handcrafted(df_split, split_name):
    cache = DATA_PROC / f"handcrafted_{split_name}.parquet"
    if cache.exists():
        print(f"  cache hit: {cache}")
        return pd.read_parquet(cache)
    t0 = time.time()
    texts = df_split["text_clean"].astype(str).tolist()
    print(f"  computing LIWC ({len(texts):,} textos)...")
    liwc = count_markers(texts, lexicon_path=LEX_PATH)
    print(f"  computing polarity...  ({time.time()-t0:.1f}s)")
    pol = polarity_score(texts, lexicon_path=POL_PATH)
    print(f"  computing temporal...  ({time.time()-t0:.1f}s)")
    tmp = temporal_extract(df_split).reset_index(drop=True)
    print(f"  total: {time.time()-t0:.1f}s | cols+hand: {liwc.shape[1]+pol.shape[1]+tmp.shape[1]}")
    out = pd.concat(
        [df_split.reset_index(drop=True), liwc.reset_index(drop=True),
         pol.reset_index(drop=True), tmp],
        axis=1,
    )
    print(f"  out shape: {out.shape}")
    out.to_parquet(cache)
    print(f"  saved: {cache}")
    return out

print("=== train ===")
df_train_full = build_handcrafted(df_train, "train")
print("=== val ===")
df_val_full = build_handcrafted(df_val, "val")
print("=== test ===")
df_test_full = build_handcrafted(df_test, "test")

## 3. Downsample train (50/50)

In [ ]:
df_pos = df_train_full[df_train_full["label"] == 2]
df_neg = df_train_full[df_train_full["label"] == 0].sample(n=len(df_pos), random_state=DOWNSAMPLE_SEED)
df_down = (
    pd.concat([df_pos, df_neg])
    .sample(frac=1.0, random_state=DOWNSAMPLE_SEED)
    .reset_index(drop=True)
)
print(f"Downsampled train: {len(df_down):,} | {df_down['label'].value_counts().to_dict()}")
assert len(df_down) == 424436

## 4. Pipeline TF-IDF + hand-crafted via ColumnTransformer

In [ ]:
cfg = load_config(REPO_ROOT / "configs" / "models.yaml")
tfidf_cfg = cfg.get("baseline", {}).get("tfidf", {})
logreg_cfg = cfg.get("baseline", {}).get("logreg", {})

# hand_cols: features hand-crafted pero EXCLUIR msg_per_user (data leakage user-level:
# groupby(user_id).size() mide tamaño del split, no actividad global → spuria en val/test).
hand_cols = [c for c in df_down.columns if c not in META_COLS and c != "text_clean" and c != "msg_per_user"]
print(f"hand_cols ({len(hand_cols)}): {hand_cols}")

pre = ColumnTransformer(
    [
        ("tfidf", TfidfVectorizer(
            ngram_range=tuple(tfidf_cfg.get("ngram_range", [1, 2])),
            max_features=int(tfidf_cfg.get("max_features", 15000)),
            min_df=int(tfidf_cfg.get("min_df", 2)),
            sublinear_tf=bool(tfidf_cfg.get("sublinear_tf", True)),
        ), "text_clean"),
        ("hand", Pipeline([("scale", StandardScaler())]), hand_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

pipeline = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(
        C=float(logreg_cfg.get("C", 1.0)),
        # Train ya está balanceado por downsampling → class_weight=None (no duplicar).
        # Baseline logreg_downsampled usa la misma convención (notebook 02).
        class_weight=None,
        max_iter=int(logreg_cfg.get("max_iter", 1000)),
        solver=logreg_cfg.get("solver", "lbfgs"),
        random_state=SEED,
    )),
])

## 5. Entrenamiento

In [ ]:
t0 = time.time()
X_train = df_down[["text_clean"] + hand_cols]
y_train = df_down["label"].astype(int).tolist()
print(f"Fit sobre {len(X_train):,} filas, {len(hand_cols)} handcrafted + tfidf...")
pipeline.fit(X_train, y_train)
print(f"OK — fit en {time.time()-t0:.1f}s")

## 6. Evaluación val + test

In [ ]:
def evaluate(pipe, name, n_train, balance_strategy, feature_set):
    rows = []
    for split_name, df_split in [("val", df_val_full), ("test", df_test_full)]:
        X = df_split[["text_clean"] + hand_cols]
        y_true = df_split["label"].astype(int).tolist()
        y_pred = pipe.predict(X)
        y_proba = pipe.predict_proba(X)
        m = compute(y_true, y_pred.tolist(), y_proba=y_proba.tolist())
        classes = list(pipe.classes_)
        pos_idx = classes.index(2) if 2 in classes else 1
        auc_bin = float(roc_auc_score(y_true, y_proba[:, pos_idx]))
        rows.append({
            "model": name,
            "split": split_name,
            "n_train_rows": n_train,
            "class_balance_strategy": balance_strategy,
            "feature_set": feature_set,
            "accuracy": m["accuracy"],
            "f1_macro": m["f1_macro"],
            "f1_weighted": m["f1_weighted"],
            "kappa": m["kappa"],
            "auc_binary_depressive": auc_bin,
            "f1_class_0_control": m["report"].get("0", {}).get("f1-score", np.nan),
            "f1_class_2_depressive": m["report"].get("2", {}).get("f1-score", np.nan),
            "created_at": pd.Timestamp.utcnow().isoformat(timespec="seconds"),
        })
    return rows

rows = evaluate(
    pipeline, "logreg_combined",
    n_train=len(df_down),
    balance_strategy="downsample_majority",
    feature_set="tfidf+handcrafted",
)
df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False))

## 7. Persistencia

In [ ]:
joblib.dump(pipeline, MODEL_DIR / "logreg_combined.joblib")
print(f"Modelo guardado en {MODEL_DIR / 'logreg_combined.joblib'}")

unified_csv = TAB_DIR / "all_models_metrics.csv"
df_existing = pd.read_csv(unified_csv)
df_combined = pd.concat([df_existing, df_metrics], ignore_index=True)
df_combined.to_csv(unified_csv, index=False)
print(f"Métricas unificadas actualizadas: {unified_csv}")
assert pipeline.classes_.tolist() == [0, 2]
assert df_metrics["auc_binary_depressive"].notna().all()
print("OK — artefactos consistentes")

## 8. Confusion matrix + leaderboard acumulado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for idx, (split_name, df_split) in enumerate([("val", df_val_full), ("test", df_test_full)]):
    X = df_split[["text_clean"] + hand_cols]
    y_true = df_split["label"].astype(int).tolist()
    y_pred = pipeline.predict(X)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 2])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=[0, 2], yticklabels=[0, 2])
    axes[idx].set_title(f"logreg_combined — {split_name}")
    axes[idx].set_xlabel("Predicho")
    axes[idx].set_ylabel("Real")
plt.tight_layout()
fig_path = FIG_DIR / "combined_confusion_matrix.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Figura guardada en {fig_path}")

In [ ]:
print("\n=== Leaderboard (test) ===")
df_all = pd.read_csv(TAB_DIR / "all_models_metrics.csv")
df_test = df_all[df_all["split"] == "test"].sort_values("f1_macro", ascending=False)
print(df_test[["model","feature_set","f1_macro","auc_binary_depressive","f1_class_2_depressive"]].to_string(index=False))

## 9. Conclusiones

- **TF-IDF + 19 features hand-crafted (LIWC + polarity + temporal)** en `logreg_combined`.
- **Schema unificado**: `reports/tables/all_models_metrics.csv` acumula baseline + combined + (próximo) BETO.
- **Cache parquet** de features hand-crafted en `data/processed/handcrafted_*.parquet` para reusar.
- **Próximo paso**: BETO fine-tuning en `notebooks/04_beto/02_finetune_beto_local.ipynb`.